In [1]:
from agents import Agent, Runner, function_tool
import asyncio

In [2]:
from dotenv import load_dotenv
import pathlib
import os

In [3]:
load_dotenv()

True

In [4]:
agent = Agent(
    name="Assistant",
    instructions=(
        "You are a helpfull assisstant."
    ),
    model="gpt-4o-mini",
    tools=[]
)

In [5]:
output = await Runner.run(agent, input="How tall is Mount Everest")

In [6]:
output.final_output

'Mount Everest is approximately 8,848.86 meters (29,031.7 feet) tall. This measurement was confirmed by a 2020 survey conducted by China and Nepal.'

### Tool Demo

In [7]:
@function_tool
def echo(text: str, repeat: int = 1) -> str:
    """Repeat the given text.
    
    Args:
        text: The phrase to repeat.
        repeat: How many times to repeat (default 1).
    """
    return " ".join([text] * max(1, repeat))

In [8]:
# ---- Agent ------------------------------------------------------------------
demo = Agent(
    name="Tiny Tool Demo",
    instructions=(
        "You are a concise assistant. "
        "When the user asks to transform or repeat text, CALL THE echo TOOL. "
        "Do not guess tool outputs—only report what the tool returns."
    ),
    model="gpt-4o-mini",
    tools=[echo],
)

In [9]:
user_input = "Use the echo tool to repeat 'hello agents' two times."
result = await Runner.run(demo, user_input)
print("FINAL OUTPUT:\n", result.final_output)

FINAL OUTPUT:
 hello agents hello agents


### Brave Search Tool

In [10]:
# main.py
import os
import asyncio
import requests
from typing import List, Dict, Optional
from agents import Agent, Runner, function_tool

BRAVE_WEB_URL = "https://api.search.brave.com/res/v1/web/search"

In [11]:
api_key = os.getenv("BRAVE_SEARCH_API_KEY")

if not api_key:
    raise RuntimeError("Missing BRAVE_SEARCH_API_KEY in environment.")


In [12]:
def brave_web_search(
    q: str,
    count: int = 5,
    country: str = "us",
    search_lang: str = "en",
    freshness: Optional[str] = None,  # e.g. "pd", "pw", "pm", "py" or "YYYY-MM-DDtoYYYY-MM-DD"
) -> List[Dict[str, str]]:
    """
    Search the web via Brave and return top results (title, url, snippet).

    Params mirror Brave docs:
      - q (required): search query.
      - count (<=20): number of web results.
      - country (2-letter), search_lang, freshness (optional window).
    """
    api_key = os.getenv("BRAVE_SEARCH_API_KEY")
    if not api_key:
        raise RuntimeError("Missing BRAVE_SEARCH_API_KEY in environment.")

    params = {
        "q": q,
        "count": max(1, min(count, 20)),     # Brave caps web count at 20
        "country": country,
        "search_lang": search_lang,
        "result_filter": "web",               # only web results
        "text_decorations": "false",          # plain snippets
    }
    if freshness:
        params["freshness"] = freshness

    headers = {
        "Accept": "application/json",
        "Accept-Encoding": "gzip",
        "X-Subscription-Token": api_key,      # required auth header
    }

    resp = requests.get(BRAVE_WEB_URL, headers=headers, params=params, timeout=15)
    resp.raise_for_status()
    data = resp.json()

    # Brave returns web results under data["web"]["results"] with title/url/description
    out: List[Dict[str, str]] = []
    for item in (data.get("web", {}) or {}).get("results", []) or []:
        out.append({
            "title": item.get("title", "")[:300],
            "url": item.get("url", ""),
            "snippet": item.get("description", "")[:500],
        })
        if len(out) >= params["count"]:
            break

    return out or [{"title": "No results", "url": "", "snippet": ""}]


brave_web_search(q="Amrita Rao", count=3, country="in", search_lang="en")

[{'title': 'Amrita Rao - Wikipedia',
  'url': 'https://en.wikipedia.org/wiki/Amrita_Rao',
  'snippet': 'Amrita Rao (born 7 June 1981) is an Indian actress who primarily works in Hindi films. Known for her quintessential girl-next-door portrayals, Rao is the recipient of several accolades including an IIFA Award and two Stardust Awards, along with nominations for two Filmfare Awards.'},
 {'title': 'AMRITA RAO 🇮🇳 (@amrita_rao_insta) • Instagram photos and videos',
  'url': 'https://www.instagram.com/amrita_rao_insta/',
  'snippet': '2M Followers, 23 Following, 914 Posts - See Instagram photos and videos from AMRITA RAO 🇮🇳 (@amrita_rao_insta)'},
 {'title': 'अमृता राव - विकिपीडिया',
  'url': 'https://translate.google.com/translate?u=https%3A%2F%2Fen.wikipedia.org%2Fwiki%2FAmrita_Rao&hl=hi&sl=en&tl=hi&client=srp',
  'snippet': 'Life Ho Toh Aisi! alongside Shahid ... year, Rao took the lead role in the John Matthew Matthan drama Shikhar, in which she portrayed Madhvi.[32] She then appeared i

In [13]:
@function_tool
def brave_web_search(
    q: str,
    count: int = 5,
    country: str = "us",
    search_lang: str = "en",
    freshness: Optional[str] = None,  # e.g. "pd", "pw", "pm", "py" or "YYYY-MM-DDtoYYYY-MM-DD"
) -> List[Dict[str, str]]:
    """
    Search the web via Brave and return top results (title, url, snippet).

    Params mirror Brave docs:
      - q (required): search query.
      - count (<=20): number of web results.
      - country (2-letter), search_lang, freshness (optional window).
    """
    api_key = os.getenv("BRAVE_SEARCH_API_KEY")
    if not api_key:
        raise RuntimeError("Missing BRAVE_SEARCH_API_KEY in environment.")

    params = {
        "q": q,
        "count": max(1, min(count, 20)),     # Brave caps web count at 20
        "country": country,
        "search_lang": search_lang,
        "result_filter": "web",               # only web results
        "text_decorations": "false",          # plain snippets
    }
    if freshness:
        params["freshness"] = freshness

    headers = {
        "Accept": "application/json",
        "Accept-Encoding": "gzip",
        "X-Subscription-Token": api_key,      # required auth header
    }

    resp = requests.get(BRAVE_WEB_URL, headers=headers, params=params, timeout=15)
    resp.raise_for_status()
    data = resp.json()

    # Brave returns web results under data["web"]["results"] with title/url/description
    out: List[Dict[str, str]] = []
    for item in (data.get("web", {}) or {}).get("results", []) or []:
        out.append({
            "title": item.get("title", "")[:300],
            "url": item.get("url", ""),
            "snippet": item.get("description", "")[:500],
        })
        if len(out) >= params["count"]:
            break

    return out or [{"title": "No results", "url": "", "snippet": ""}]


In [14]:
# --- Agent that uses the tool ------------------------------------------------
search_agent = Agent(
    name="Brave Searcher",
    instructions=(
        "You search the web. When the user asks to look something up, "
        "CALL the brave_web_search tool. Then return a concise bullet list:\n"
        "- Title — URL\n  One-line takeaway."
    ),
    model="gpt-4o-mini",
    tools=[brave_web_search],
)

In [15]:
# Example user ask that *forces* tool use
user_msg = "Search the web for: 'python virtualenv tutorial' (top 5)."
result = await Runner.run(search_agent, user_msg)
print(result.final_output)

Here are the top 5 results for "python virtualenv tutorial":

- **12. Virtual Environments and Packages — Python 3.13.7 documentation** — [docs.python.org](https://docs.python.org/3/tutorial/venv.html)  
  Official documentation covering virtual environments in Python.

- **How to use Python virtualenv - Python Tutorial** — [pythonbasics.org](https://pythonbasics.org/virtualenv/)  
  Explains how virtualenv creates isolated environments for different Python applications.

- **A Complete Guide to Python Virtual Environments (2022) – Dataquest** — [dataquest.io](https://www.dataquest.io/blog/a-complete-guide-to-python-virtual-environments/)  
  Comprehensive tutorial on creating and managing virtual environments.

- **Python venv: How To Create, Activate, Deactivate, And Delete • Python Land Tutorial** — [python.land](https://python.land/virtual-environments/virtualenv)  
  Instructions for creating, activating, and deleting Python virtual environments across different operating systems.

### Scrape Tool

In [16]:
# main.py
import os
import time
import json
import asyncio
import hashlib
from typing import Dict, Any, Optional
from urllib.parse import urlparse

import requests
import trafilatura
from agents import Agent, Runner, function_tool

In [17]:
# ---- Simple, latency-conscious scraper --------------------------------------
def scrape_url(
    url: str,
    max_chars: int = 8000,
    with_links: bool = True,
    timeout_s: float = 12.0,
) -> Dict[str, Any]:
    """
    Fetch an HTML page and return main content + metadata for LLM/RAG.

    Args:
      url: Page URL (HTML pages; PDFs not supported here).
      max_chars: Truncate text to reduce tokens/latency (0 = no limit).
      with_links: Include extracted outgoing links (helps citations).
      timeout_s: Network timeout (seconds).

    Returns:
      {
        "url": str,            # original URL
        "final_url": str,      # after redirects
        "title": str|None,
        "author": str|None,
        "date": str|None,      # as provided by page (if any)
        "language": str|None,
        "text": str,           # clean main content (LLM-ready)
        "links": list[dict],   # [{'url','text'}] if available
        "word_count": int,
        "latency_ms": int,
        "sha1": str,           # hash of returned text
        "site": str,           # hostname
        "error": str|None
      }
    """
    t0 = time.time()
    headers = {
        # Polite but effective: some sites block default UA
        "User-Agent": "Mozilla/5.0 (compatible; SimpleScraper/1.0; +https://example.org/agent)",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "close",
    }

    try:
        resp = requests.get(url, headers=headers, timeout=timeout_s, allow_redirects=True)
    except requests.RequestException as e:
        return {
            "url": url, "final_url": url, "title": None, "author": None, "date": None,
            "language": None, "text": "", "links": [], "word_count": 0,
            "latency_ms": int((time.time() - t0) * 1000), "sha1": "",
            "site": urlparse(url).hostname or "", "error": f"network_error: {e}"
        }

    ctype = (resp.headers.get("Content-Type") or "").lower()
    if "text/html" not in ctype and "application/xhtml+xml" not in ctype:
        return {
            "url": url, "final_url": resp.url, "title": None, "author": None, "date": None,
            "language": None, "text": "", "links": [], "word_count": 0,
            "latency_ms": int((time.time() - t0) * 1000), "sha1": "",
            "site": urlparse(resp.url).hostname or "", "error": f"unsupported_content_type: {ctype}"
        }

    html = resp.text or ""
    # High-quality main-content extraction (fast, no JS)
    extracted_json = trafilatura.extract(
        html,
        url=resp.url,                          # helps canonicalization
        output_format="json",
        include_links=with_links,
        include_formatting=True,               # preserve headings/code markers
        include_tables=True,
        favor_precision=True,                  # better precision for docs/news
    )

    if extracted_json:
        data = json.loads(extracted_json)
        title = data.get("title")
        text = data.get("text") or ""
        author = data.get("author")
        date = data.get("date")
        language = data.get("language")
        links = data.get("links") or [] if with_links else []
    else:
        # Fallback: plain text (still robust)
        title = None
        author = None
        date = None
        language = None
        links = []
        text = trafilatura.extract(html, url=resp.url, output_format="txt") or ""

    if max_chars and len(text) > max_chars:
        text = text[:max_chars].rstrip()

    word_count = len(text.split())
    sha1 = hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()

    return {
        "url": url,
        "final_url": resp.url,
        "title": title,
        "author": author,
        "date": date,
        "language": language,
        "text": text,
        "links": links,
        "word_count": word_count,
        "latency_ms": int((time.time() - t0) * 1000),
        "sha1": sha1,
        "site": urlparse(resp.url).hostname or "",
        "error": None,
    }

scrape_url(
    url="https://en.wikipedia.org/wiki/Amrita_Rao",
    max_chars=5000
)

{'url': 'https://en.wikipedia.org/wiki/Amrita_Rao',
 'final_url': 'https://en.wikipedia.org/wiki/Amrita_Rao',
 'title': None,
 'author': None,
 'date': None,
 'language': None,
 'text': "Amrita Rao | |\n|---|---|\n| Born |\n| 7 June 1981\n| Alma mater |\n|\n[[3]](https://en.wikipedia.org#cite_note-3)[Preetika Rao](https://en.wikipedia.org/wiki/Preetika_Rao)(sister)Amrita Rao (born 7 June 1981) is an Indian actress who primarily works in [Hindi films](https://en.wikipedia.org/wiki/Hindi_cinema). Known for her quintessential girl-next-door portrayals, Rao is the recipient of several accolades including an [IIFA Award](https://en.wikipedia.org/wiki/IIFA_Awards) and two [Stardust Awards](https://en.wikipedia.org/wiki/Stardust_Awards), along with nominations for two [Filmfare Awards](https://en.wikipedia.org/wiki/Filmfare_Awards).[[4]](https://en.wikipedia.org#cite_note-girl-4)[[5]](https://en.wikipedia.org#cite_note-5)\nRao made her acting debut with Ab Ke Baras (2002), which earned her a 

In [18]:
# ---- Simple, latency-conscious scraper --------------------------------------
@function_tool
def scrape_url(
    url: str,
    max_chars: int = 8000,
    with_links: bool = True,
    timeout_s: float = 12.0,
) -> Dict[str, Any]:
    """
    Fetch an HTML page and return main content + metadata for LLM/RAG.

    Args:
      url: Page URL (HTML pages; PDFs not supported here).
      max_chars: Truncate text to reduce tokens/latency (0 = no limit).
      with_links: Include extracted outgoing links (helps citations).
      timeout_s: Network timeout (seconds).

    Returns:
      {
        "url": str,            # original URL
        "final_url": str,      # after redirects
        "title": str|None,
        "author": str|None,
        "date": str|None,      # as provided by page (if any)
        "language": str|None,
        "text": str,           # clean main content (LLM-ready)
        "links": list[dict],   # [{'url','text'}] if available
        "word_count": int,
        "latency_ms": int,
        "sha1": str,           # hash of returned text
        "site": str,           # hostname
        "error": str|None
      }
    """
    t0 = time.time()
    headers = {
        # Polite but effective: some sites block default UA
        "User-Agent": "Mozilla/5.0 (compatible; SimpleScraper/1.0; +https://example.org/agent)",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "close",
    }

    try:
        resp = requests.get(url, headers=headers, timeout=timeout_s, allow_redirects=True)
    except requests.RequestException as e:
        return {
            "url": url, "final_url": url, "title": None, "author": None, "date": None,
            "language": None, "text": "", "links": [], "word_count": 0,
            "latency_ms": int((time.time() - t0) * 1000), "sha1": "",
            "site": urlparse(url).hostname or "", "error": f"network_error: {e}"
        }

    ctype = (resp.headers.get("Content-Type") or "").lower()
    if "text/html" not in ctype and "application/xhtml+xml" not in ctype:
        return {
            "url": url, "final_url": resp.url, "title": None, "author": None, "date": None,
            "language": None, "text": "", "links": [], "word_count": 0,
            "latency_ms": int((time.time() - t0) * 1000), "sha1": "",
            "site": urlparse(resp.url).hostname or "", "error": f"unsupported_content_type: {ctype}"
        }

    html = resp.text or ""
    # High-quality main-content extraction (fast, no JS)
    extracted_json = trafilatura.extract(
        html,
        url=resp.url,                          # helps canonicalization
        output_format="json",
        include_links=with_links,
        include_formatting=True,               # preserve headings/code markers
        include_tables=True,
        favor_precision=True,                  # better precision for docs/news
    )

    if extracted_json:
        data = json.loads(extracted_json)
        title = data.get("title")
        text = data.get("text") or ""
        author = data.get("author")
        date = data.get("date")
        language = data.get("language")
        links = data.get("links") or [] if with_links else []
    else:
        # Fallback: plain text (still robust)
        title = None
        author = None
        date = None
        language = None
        links = []
        text = trafilatura.extract(html, url=resp.url, output_format="txt") or ""

    if max_chars and len(text) > max_chars:
        text = text[:max_chars].rstrip()

    word_count = len(text.split())
    sha1 = hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()

    return {
        "url": url,
        "final_url": resp.url,
        "title": title,
        "author": author,
        "date": date,
        "language": language,
        "text": text,
        "links": links,
        "word_count": word_count,
        "latency_ms": int((time.time() - t0) * 1000),
        "sha1": sha1,
        "site": urlparse(resp.url).hostname or "",
        "error": None,
    }

In [19]:
# ---- Agent that uses the scraper --------------------------------------------
scraper_agent = Agent(
    name="Web Scraper",
    instructions=(
        "You extract information from web pages. "
        "When the user provides a URL (or asks to scrape), CALL the scrape_url tool. "
        "Then return a compact summary:\n"
        "- Title (if any)\n- Site and date\n- 4–6 bullet key points\n- 'Source: <final_url>'\n"
        "If the page has very little text, say so and include the first 300 characters."
    ),
    tools=[scrape_url],
)

In [20]:
# ---- Demo run ----------------------------------------------------------------

# Example prompt that forces tool-use
user_msg = (
    "Scrape this whole page (30000 chars) and summarise the profile in a table: https://en.wikipedia.org/wiki/Amrita_Rao"
    "Show example implementation if required, if not then ignore"
)
result = await Runner.run(scraper_agent, user_msg)
print(result.final_output)

Here's a summary of Amrita Rao's Wikipedia profile in a tabular format:

| **Attribute**       | **Details**                                                                                                                                                      |
|---------------------|------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **Name**            | Amrita Rao                                                                                                                                                       |
| **Date of Birth**   | 7 June 1981                                                                                                                                                      |
| **Occupation**      | Actress                                                                                                                                                    

# Search+Scrape Agent

In [23]:
search_agent = Agent(
    name="Brave Searcher",
    instructions=(
        "You search the web. When the user asks to look something up, "
        "CALL the brave_web_search tool. Then return a concise bullet list:\n"
        "- Title — URL\n  One-line takeaway."
        "Markdown URL Link"
    ),
    model="gpt-4o-mini",
    tools=[brave_web_search],
)
scraper_agent = Agent(
    name="Web Scraper",
    instructions=(
        "You are given a list of URL links to web pages"
        "You extract information from web pages."
        "When the user provides a URLs, CALL the scrape_url tool. "
        "Then return a summary one by one for each URL:\n"
        "- Title (if any)\n- Site and date\n- bullet key points\n- 'Source: <final_url>'\n"
    ),
    tools=[scrape_url],
    model="gpt-4o-mini",
)


async def Orchestration():
    print("Ask the topic.")
    user = input("> ").strip()
    search_result = await Runner.run(search_agent, "I want to know about: "+user)
    print(search_result.final_output)
    print("\n\n")
    extracted_urls = extract_urls(search_result.final_output)
    print(extracted_urls)
    print("\n\n")
    scrape_result = await Runner.run(scraper_agent, "Following are the extracted URLs: " + str(extracted_urls))
    print(scrape_result.final_output)

await Orchestration()

Ask the topic.


>  Amrita Rao


Here's a brief overview of Amrita Rao:

- **Amrita Rao - Wikipedia** — [Link](https://en.wikipedia.org/wiki/Amrita_Rao)  
  Amrita Rao is an Indian actress born on June 7, 1981, primarily known for her roles in Hindi films. She has received multiple awards including an IIFA Award and two Stardust Awards.

- **AMRITA RAO 🇮🇳 (@amrita_rao_insta) • Instagram** — [Link](https://www.instagram.com/amrita_rao_insta/)  
  Official Instagram of Amrita Rao, featuring her posts and updates with 2M followers.

- **Amrita Rao | Actress (IMDb)** — [Link](https://www.imdb.com/name/nm1182255/)  
  Profile detailing her career, including her notable film "Vivah," and background information.

- **Amrita Rao | Facebook** — [Link](https://www.facebook.com/AmritaRaoActress/)  
  Official Facebook page with updates and fan interaction, showcasing her popularity as Bollywood's "Girl Next Door."

- **AMRITA RAO on Instagram** — [Link](https://www.instagram.com/reel/DKBvtCPo6NX/)  
  Latest reel from her Instag

In [22]:
import re
from urllib.parse import urlparse

# 1) Markdown links: [label](https://example.com)
MD_LINK_RE = re.compile(r"\[[^\]]+\]\(\s*(https?://[^)\s]+)\s*\)", re.IGNORECASE)

# 2) Bare URLs that are NOT inside [] or () just before them
#    Stop at whitespace or ] ) < > " '
BARE_URL_RE = re.compile(r"(?<!\()(?<!\[)(https?://[^\s\]\)<>\"']+)", re.IGNORECASE)

TRAILING_CHARS = ".,);]>”’"

def extract_urls(text: str):
    """Return unique http/https URLs in first-seen order (Markdown + bare)."""
    urls, seen = [], set()

    # Pass 1: Markdown links (the URL in (...))
    for m in MD_LINK_RE.finditer(text):
        url = m.group(1).rstrip(TRAILING_CHARS)
        if urlparse(url).scheme in ("http", "https") and url not in seen:
            seen.add(url)
            urls.append(url)

    # Pass 2: Bare URLs (avoid those that are part of a Markdown [label])
    for m in BARE_URL_RE.finditer(text):
        url = m.group(1).rstrip(TRAILING_CHARS)
        if urlparse(url).scheme in ("http", "https") and url not in seen:
            seen.add(url)
            urls.append(url)

    return urls


# --- Example ---
text = """
Here are the top 5 tutorials on Python virtual environments:

- **12. Virtual Environments and Packages — Python 3.13.7 documentation** — [https://docs.python.org/3/tutorial/venv.html](https://docs.python.org/3/tutorial/venv.html)  
  Official documentation on using virtual environments in Python.

- **How to use Python virtualenv - Python Tutorial** — [https://pythonbasics.org/virtualenv/](https://pythonbasics.org/virtualenv/)  
  A beginner-friendly guide explaining the purpose and usage of virtualenv.

- **A Complete Guide to Python Virtual Environments (2022) – Dataquest** — [https://www.dataquest.io/blog/a-complete-guide-to-python-virtual-environments/](https://www.dataquest.io/blog/a-complete-guide-to-python-virtual-environments/)  
  Comprehensive guide covering creation, activation, and management of virtual environments.

- **Python venv: How To Create, Activate, Deactivate, And Delete • Python Land Tutorial** — [https://python.land/virtual-environments/virtualenv](https://python.land/virtual-environments/virtualenv)  
  Instructions on creating and using Python venv across different operating systems.

- **Python Virtual Environment Tutorial · GitHub** — [https://gist.github.com/ryumada/c22133988fd1c22a66e4ed1b23eca233](https://gist.github.com/ryumada/c22133988fd1c22a66e4ed1b23eca233)  
  A GitHub Gist sharing a concise tutorial on Python virtual environments.
"""

print(extract_urls(text))
# -> ['https://docs.python.org/3/tutorial/venv.html',
#     'https://pythonbasics.org/virtualenv/',
#     'https://www.dataquest.io/blog/a-complete-guide-to-python-virtual-environments/',
#     'https://python.land/virtual-environments/virtualenv',
#     'https://gist.github.com/ryumada/c22133988fd1c22a66e4ed1b23eca233']


['https://docs.python.org/3/tutorial/venv.html', 'https://pythonbasics.org/virtualenv/', 'https://www.dataquest.io/blog/a-complete-guide-to-python-virtual-environments/', 'https://python.land/virtual-environments/virtualenv', 'https://gist.github.com/ryumada/c22133988fd1c22a66e4ed1b23eca233']


In [58]:
# scraper_agent = Agent(
#     name="Web Scraper",
#     instructions=(
#         "You are given a list of URL links to web pages"
#         "You extract information from web pages."
#         "When the user provides a URLs, CALL the scrape_url tool. "
#         "Then return a summary one by one for each URL:\n"
#         "- Title (if any)\n- Site and date\n- bullet key points\n- 'Source: <final_url>'\n"
#     ),
#     model="gpt-4o-mini",
#     tools=[scrape_url],
# )

scraper_agent = Agent(
    name="Web Scraper",
    instructions=(
        "You extract information from web pages. "
        "When the user provides a URL (or asks to scrape), CALL the scrape_url tool. "
        "Then return a compact summary:\n"
        "- Title (if any)\n- Site and date\n- 4–6 bullet key points\n- 'Source: <final_url>'\n"
        "If the page has very little text, say so and include the first 300 characters."
    ),
    tools=[scrape_url],
)
await Runner.run(scraper_agent, "Hi")

AttributeError: 'function' object has no attribute 'name'